# Notebook 08 — L3 Category Classification
## Scaling to the Granular Taxonomy Level (48 classes)

**Project**: Cloud-Based ITSM Ticket Classification Platform Using Fine-Tuned Transformer Models  
**Author**: Mohamed A. Elbaz  

---

### Objective

This notebook represents the final step in category classification. We extend the model to **Level 3 (L3)**, which contains **48 classes**. This is the most challenging task due to label granularity and potential class imbalance.

**Workflow**:
1. Load the processed data with L1, L2, and L3 labels.
2. Fine-tune MarBERTv2 with three heads: `label_l1`, `label_l2`, and `label_l3`.
3. Analyze performance across the full hierarchy.
4. Evaluate the "sparsity challenge" — how well does the model handle the 48 granular classes?

> **Note on MarBERT Load Report**: When loading the model, you may see "UNEXPECTED" keys in the load report (e.g., `cls.predictions.*`). This is **normal behavior**. MarBERTv2 was pre-trained with Masked Language Modeling (MLM) and Next Sentence Prediction (NSP) heads. Since we are using it for classification, these heads are discarded and replaced by our custom task heads. You can safely ignore these warnings.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import pandas as pd
import pickle
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

from arabic_itsm.data.preprocessing import ArabicTextNormalizer
from arabic_itsm.data.dataset import ITSMDataset
from arabic_itsm.models.classifier import MarBERTClassifier
from arabic_itsm.utils.metrics import compute_classification_metrics

MODEL_NAME = "UBC-NLP/MARBERTv2"
TASKS      = ['l1', 'l2', 'l3']
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIG_DIR    = Path('../results/figures')
METRICS_DIR = Path('../results/metrics')

## 1. Data Preparation

In [ ]:
DATA_DIR = Path('../data/processed')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')

with open(DATA_DIR / 'label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
normalizer = ArabicTextNormalizer()

train_ds = ITSMDataset(train_df, tokenizer, normalizer, label_encoders, tasks=TASKS)
val_ds   = ITSMDataset(val_df, tokenizer, normalizer, label_encoders, tasks=TASKS)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"Tasks: {TASKS}")
print(f"Classes: {train_ds.num_classes}")

## 2. Joint Training (L1 + L2 + L3)

In [ ]:
model = MarBERTClassifier(MODEL_NAME, train_ds.num_classes).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

best_l3_f1 = 0.0
OUT_DIR = Path('../models/marbert_l3_best')
OUT_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(1, 6):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = {f"label_{t}": batch[f"label_{t}"].to(DEVICE) for t in TASKS}

        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            out = model(ids, mask, **labels)
            loss = out["loss"]

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    # Validation
    model.eval()
    all_preds = {t: [] for t in TASKS}
    all_labels = {t: [] for t in TASKS}
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            out = model(ids, mask)
            for t in TASKS:
                all_preds[t].extend(torch.argmax(out[f"logits_{t}"], dim=-1).cpu().numpy())
                all_labels[t].extend(batch[f"label_{t}"].numpy())

    # Compute Metrics
    res_l1 = compute_classification_metrics(all_labels['l1'], all_preds['l1'])
    res_l2 = compute_classification_metrics(all_labels['l2'], all_preds['l2'])
    res_l3 = compute_classification_metrics(all_labels['l3'], all_preds['l3'])
    
    print(f"Epoch {epoch} | Loss: {train_loss/len(train_loader):.4f} | L1-F1: {res_l1['macro_f1']:.4f} | L2-F1: {res_l2['macro_f1']:.4f} | L3-F1: {res_l3['macro_f1']:.4f}")

    if res_l3['macro_f1'] > best_l3_f1:
        best_l3_f1 = res_l3['macro_f1']
        torch.save(model.heads.state_dict(), OUT_DIR / "heads.pt")
        model.encoder.save_pretrained(str(OUT_DIR))
        print("  ✓ Saved best L3 checkpoint")

## 3. Results & Visualization
Given that L3 has 48 classes, a raw confusion matrix is hard to read. We visualize the **Top 10 Most Challenging Classes** and the overall performance decay across hierarchy levels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

# 1. Overall hierarchy decay
levels = ['L1 (6 classes)', 'L2 (16 classes)', 'L3 (48 classes)']
f1_scores = [res_l1['macro_f1'], res_l2['macro_f1'], res_l3['macro_f1']]

plt.figure(figsize=(8, 5))
sns.barplot(x=levels, y=f1_scores, palette='viridis')
plt.ylim(0, 1)
plt.title('Macro-F1 Performance Across Taxonomy Levels', fontweight='bold')
for i, v in enumerate(f1_scores):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontweight='bold')
plt.savefig(FIG_DIR / '08_hierarchy_performance_decay.png', bbox_inches='tight')
plt.show()

# 2. Per-class report snippet for L3
report_l3 = classification_report(all_labels['l3'], all_preds['l3'], 
                                 target_names=label_encoders['l3'].classes_, output_dict=True)
df_report_l3 = pd.DataFrame(report_l3).transpose()
df_report_l3.to_csv(METRICS_DIR / '08_l3_per_class_report.csv')

print("\n--- L3 Classification Summary ---")
print("As expected, the Macro-F1 drops as the number of classes increases. This is a common pattern in hierarchical classification where data sparsity becomes an issue at the leaf nodes.")